In [18]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

In [7]:
# Load ratings
ratings = pd.read_csv("ml-32m/ratings.csv")

# Group by movie to get the average rating and total number of ratings
movie_stats = ratings.groupby("movieId")["rating"].agg(["mean", "count"]).reset_index()
movie_stats.rename(columns={"mean": "avg_rating", "count": "num_ratings"}, inplace=True)

# EXCLUDE movies with fewer than 50 ratings
popular_movies_stats = movie_stats[movie_stats["num_ratings"] >= 50]

display(popular_movies_stats.head())


,movieId,avg_rating,num_ratings
0,1,3.897438,68997
1,2,3.275758,28904
2,3,3.139447,13134
3,4,2.845331,2806
4,5,3.059602,13154


In [10]:
# Load movies
movies = pd.read_csv("ml-32m/movies.csv")

# Merge movies with our popular_movies_stats
# We use an 'inner' join, so any movie NOT in popular_movies_stats gets dropped instantly!
movies = movies.merge(popular_movies_stats, on="movieId", how="inner")

# Clean the genres text
movies["genres"] = movies["genres"].replace("(no genres listed)", "")
movies["genres_clean"] = movies["genres"].str.replace("|", " ", regex=False).str.lower()

# Reset index because we dropped a lot of rows during the merge
movies = movies.reset_index(drop=True)

display(movies[["title", "genres_clean", "avg_rating", "num_ratings"]].head())


,title,genres_clean,avg_rating,num_ratings
0,Toy Story (1995),adventure animation children comedy fantasy,3.897438,68997
1,Jumanji (1995),adventure children fantasy,3.275758,28904
2,Grumpier Old Men (1995),comedy romance,3.139447,13134
3,Waiting to Exhale (1995),comedy drama romance,2.845331,2806
4,Father of the Bride Part II (1995),comedy,3.059602,13154


In [12]:
# Initialize and apply TF-IDF
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["genres_clean"])

print(f"New TF-IDF Matrix Shape (Filtered): {tfidf_matrix.shape}")


New TF-IDF Matrix Shape (Filtered): (16034, 21)


In [14]:
# Reverse mapping
indices = pd.Series(movies.index, index=movies["title"]).drop_duplicates()

def get_recommendations(title, top_n=10):
    if title not in indices:
        return f"Error: '{title}' not found. It might have fewer than 50 ratings."
    
    idx = indices[title]
    
    # Calculate similarity for just this movie against all others
    sim_scores_array = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    
    # Copy the dataframe so we don't modify the original
    recommended = movies.copy()
    
    # Add the similarity scores as a new column
    recommended["similarity_score"] = sim_scores_array
    
    # Drop the movie we searched for (so it doesn't recommend itself)
    recommended = recommended[recommended.index != idx]
    
    # SORTING MAGIC: Sort by similarity FIRST, then by avg_rating SECOND (descending)
    recommended = recommended.sort_values(
        by=["similarity_score", "avg_rating"], 
        ascending=[False, False]
    )
    
    # Return the top N results, showing only relevant columns
    return recommended[["title", "genres", "similarity_score", "avg_rating", "num_ratings"]].head(top_n)


In [16]:
test_movie = "Toy Story (1995)"
display(get_recommendations(test_movie))


,title,genres,similarity_score,avg_rating,num_ratings
15954,Puss in Boots: The Last Wish (2022),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.921983,1301
15488,Soul (2020),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.911948,4486
4385,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.0,3.837442,46036
13918,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.816381,8278
2786,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.812043,32683
3585,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0,3.650783,11437
15622,Luca (2021),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.638158,1216
15338,Onward (2020),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.596420,1732
12362,"Boxtrolls, The (2014)",Adventure|Animation|Children|Comedy|Fantasy,1.0,3.334826,893
15997,The Super Mario Bros. Movie (2023),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.288104,538


In [20]:
# Save the cleaned dataframe
movies.to_pickle("movies_df.pkl")

# Save the TF-IDF matrix
joblib.dump(tfidf_matrix, "tfidf_matrix.pkl")


['tfidf_matrix.pkl']